In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the clean, merged master dataset from Notebook 1
df = pd.read_parquet("clean_master_energy_weather.parquet")

# Quick check to make sure datetimes and columns loaded correctly
print(f"Loaded master dataset successfully with {len(df):,} rows.")
print(df.dtypes)
df.head()

Loaded master dataset successfully with 483,731 rows.
utc_time                     datetime64[ns, UTC]
power_consumed_kw                        float64
power_produced_kw                        float64
power_consumed_phase_1_kw                float64
power_produced_phase_1_kw                float64
solar_output_kw                          float64
pv_all_power_kw                          float64
pv1_power_kw                             float64
supplier                                  object
Station                                    int64
Date                                       int64
Time                                       int64
Temperature                              float64
Sunshine_hrs                             float64
Global_radiation                           int64
Cloud_cover                               object
day_of_week                               object
is_weekend                                  bool
month                                      int32
hour           

,utc_time,power_consumed_kw,power_produced_kw,power_consumed_phase_1_kw,power_produced_phase_1_kw,solar_output_kw,pv_all_power_kw,pv1_power_kw,supplier,Station,Date,Time,Temperature,Sunshine_hrs,Global_radiation,Cloud_cover,day_of_week,is_weekend,month,hour
0,2025-10-16 20:15:00+00:00,0.528200,0.0,0.527500,0.0,NaN,NaN,NaN,Vattenfall,240,20251016,20,12.9,0.0,0,8,Thursday,False,10,20
1,2025-10-16 20:16:00+00:00,0.526660,0.0,0.529296,0.0,NaN,NaN,NaN,Vattenfall,240,20251016,20,12.9,0.0,0,8,Thursday,False,10,20
2,2025-10-16 20:17:00+00:00,0.508875,0.0,0.506186,0.0,NaN,NaN,NaN,Vattenfall,240,20251016,20,12.9,0.0,0,8,Thursday,False,10,20
3,2025-10-16 20:18:00+00:00,0.520357,0.0,0.521571,0.0,NaN,NaN,NaN,Vattenfall,240,20251016,20,12.9,0.0,0,8,Thursday,False,10,20
4,2025-10-16 20:19:00+00:00,0.524898,0.0,0.526386,0.0,NaN,NaN,NaN,Vattenfall,240,20251016,20,12.9,0.0,0,8,Thursday,False,10,20


## Baseline Reality: Calculate Consumption Costs & Solar Revenues for Vattenfall & Greenchoice

<i>Vattenfall</i><br>
Fixed consumption rate1 (off-peak): 0.25971<br>
Fixed consumption rate2 (peak/normal): 0.29383<br>

Solar feed-in compensation: 0.168

<i>Greenchoice</i><br>
Fixed consumption rate1 (off-peak): 0.23443<br>
Fixed consumption rate2 (peak/normal): 0.22233<br>

Solar feed-in compensation minus terugleverkosten: 0.013

In [3]:
# Computes the financial totals (in Euros) for both consumption cost and solar revenue on a minute-by-minute basis.
# Converts kW to kWh, checks peak/off-peak hours, assigns supplier-specific import rates, and computes complex solar revenue logic 
# (including Greenchoice's net feed-in calculations)

def calculate_actual_costs_split(row):
    # Total consumption and production per minute (kWh)
    kwh_consumed = row.get('power_consumed_kw', 0.0) / 60.0
    kwh_produced = row.get('power_produced_kw', 0.0) / 60.0
    
    # Determine if current minute falls in Peak (Normaal: 07:00 - 23:00 UTC/Local) or Off-Peak
    hour = row['utc_time'].hour
    is_peak = 7 <= hour < 23
    
    supplier = row.get('supplier', 'Vattenfall')
    
    if supplier == 'Vattenfall':
        rate = 0.29383 if is_peak else 0.25971
        solar_revenue = kwh_produced * 0.16800
    else:  # Greenchoice
        rate = 0.22233 if is_peak else 0.23443
        net_solar_rate = 0.15 - 0.13613
        solar_revenue = kwh_produced * net_solar_rate
        
    consumption_cost = kwh_consumed * rate
    return pd.Series([consumption_cost, solar_revenue])

# Re-apply the function across the dataframe
df[['actual_consumption_cost_eur', 'actual_solar_revenue_eur']] = df.apply(calculate_actual_costs_split, axis=1)

# Check the results again
print(df[['actual_consumption_cost_eur', 'actual_solar_revenue_eur']].head())
print("\nTotal Consumption Cost (€):", df['actual_consumption_cost_eur'].sum())
print("Total Solar Revenue (€):", df['actual_solar_revenue_eur'].sum())

   actual_consumption_cost_eur  actual_solar_revenue_eur
0                     0.002587                       0.0
1                     0.002579                       0.0
2                     0.002492                       0.0
3                     0.002548                       0.0
4                     0.002571                       0.0

Total Consumption Cost (€): 1098.4706264170213
Total Solar Revenue (€): 17.452711900152984


In [4]:
# Determines only the unit price rate (€/kWh) for energy consumption based on the time of day and supplier.

def get_actual_rate(row):
  hour = row['utc_time'].hour
  is_peak = 7 <= hour < 23
  supplier = row.get('supplier', 'Vattenfall')

  if supplier == 'Vattenfall':
    return 0.29383 if is_peak else 0.25971
  else:  # Greenchoice
    return 0.22233 if is_peak else 0.23443


# Create the new column in your dataframe
df['actual_rate'] = df.apply(get_actual_rate, axis=1)

In [5]:
df.head()

,utc_time,power_consumed_kw,power_produced_kw,power_consumed_phase_1_kw,power_produced_phase_1_kw,solar_output_kw,pv_all_power_kw,pv1_power_kw,supplier,Station,...,Sunshine_hrs,Global_radiation,Cloud_cover,day_of_week,is_weekend,month,hour,actual_consumption_cost_eur,actual_solar_revenue_eur,actual_rate
0,2025-10-16 20:15:00+00:00,0.528200,0.0,0.527500,0.0,NaN,NaN,NaN,Vattenfall,240,...,0.0,0,8,Thursday,False,10,20,0.002587,0.0,0.29383
1,2025-10-16 20:16:00+00:00,0.526660,0.0,0.529296,0.0,NaN,NaN,NaN,Vattenfall,240,...,0.0,0,8,Thursday,False,10,20,0.002579,0.0,0.29383
2,2025-10-16 20:17:00+00:00,0.508875,0.0,0.506186,0.0,NaN,NaN,NaN,Vattenfall,240,...,0.0,0,8,Thursday,False,10,20,0.002492,0.0,0.29383
3,2025-10-16 20:18:00+00:00,0.520357,0.0,0.521571,0.0,NaN,NaN,NaN,Vattenfall,240,...,0.0,0,8,Thursday,False,10,20,0.002548,0.0,0.29383
4,2025-10-16 20:19:00+00:00,0.524898,0.0,0.526386,0.0,NaN,NaN,NaN,Vattenfall,240,...,0.0,0,8,Thursday,False,10,20,0.002571,0.0,0.29383


## Calculate Feed-in Penalties (Terugleverkosten)

In [6]:
# Convert minute power (kW) to energy (kWh) for the summary aggregation
df['energy_consumed_kwh'] = df['power_consumed_kw'] / 60.0
df['energy_produced_kwh'] = df['power_produced_kw'] / 60.0

# Group by supplier to see total consumption, solar export, and costs for each period
supplier_summary = df.groupby('supplier').agg({
    'energy_consumed_kwh': 'sum',
    'energy_produced_kwh': 'sum',
    'actual_consumption_cost_eur': 'sum',
    'actual_solar_revenue_eur': 'sum'
})

print("--- Summary by Supplier Period ---")
print(supplier_summary)

--- Summary by Supplier Period ---
             energy_consumed_kwh  energy_produced_kwh  \
supplier                                                
Greenchoice          2453.962370           608.432030   
Vattenfall           1904.121881            53.653331   

             actual_consumption_cost_eur  actual_solar_revenue_eur  
supplier                                                            
Greenchoice                   551.545383                  8.438952  
Vattenfall                    546.925243                  9.013760  


In [7]:
# Calculate Terugleverkosten (TLK) per supplier rules:
# --- Vattenfall (Oct 2025 - Feb 2026): Fixed bracket fee ---

vattenfall_export = supplier_summary.loc['Vattenfall', 'energy_produced_kwh']
vattenfall_tlk = 121.00 * (5/12) # Prorated for the 5 months

# --- Greenchoice (Feb 2026 - Sep 2026): Per-kWh fee ---
# Greenchoice charge: €0.13613 per kWh

greenchoice_export = supplier_summary.loc['Greenchoice', 'energy_produced_kwh']
greenchoice_tlk = greenchoice_export * 0.13613

total_tlk = vattenfall_tlk + greenchoice_tlk

print(f"Calculated Vattenfall TLK portion: €{vattenfall_tlk:.2f}")
print(f"Calculated Greenchoice TLK portion: €{greenchoice_tlk:.2f}")
print(f"Total Actual Terugleverkosten: €{total_tlk:.2f}")

Calculated Vattenfall TLK portion: €50.42
Calculated Greenchoice TLK portion: €82.83
Total Actual Terugleverkosten: €133.24


In [8]:
# Factor in fixed annual baseline items (Liander network fees + fixed delivery - tax credit)
# From the Vattenfall desktop research: Liander = €455.74, Fixed delivery = €107.88, Tax credit = -€635.19

net_fixed_annual = 455.74 + 107.88 - 635.19  # Total net fixed cost/credit for the year

# Calculate Final Actual Baseline Total
total_actual_consumption = supplier_summary['actual_consumption_cost_eur'].sum()
total_actual_solar_rev = supplier_summary['actual_solar_revenue_eur'].sum()

final_actual_cost = total_actual_consumption + total_tlk + net_fixed_annual - total_actual_solar_rev

print(f"\n========================================")
print(f"FINAL ACTUAL HISTORICAL ENERGY SPEND: €{final_actual_cost:.2f}")
print(f"========================================")


FINAL ACTUAL HISTORICAL ENERGY SPEND: €1142.69


In [9]:
# Apply a row-level fee based on exported solar per minute
def get_row_tlk(row):
  exported = max(
      0.0,
      (row.get('energy_produced_kw', 0.0) / 60.0)
      - (row.get('energy_consumed_kw', 0.0) / 60.0),
  )
  supplier = row.get('supplier', 'Vattenfall')
  rate = 0.13613 if supplier == 'Greenchoice' else (121.00 / (365 * 24))
  return exported * rate


df['row_terugleverkosten'] = df.apply(get_row_tlk, axis=1)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 483731 entries, 0 to 483730
Data columns (total 26 columns):
 #   Column                       Non-Null Count   Dtype              
---  ------                       --------------   -----              
 0   utc_time                     483731 non-null  datetime64[ns, UTC]
 1   power_consumed_kw            483731 non-null  float64            
 2   power_produced_kw            483731 non-null  float64            
 3   power_consumed_phase_1_kw    483731 non-null  float64            
 4   power_produced_phase_1_kw    483731 non-null  float64            
 5   solar_output_kw              100194 non-null  float64            
 6   pv_all_power_kw              73332 non-null   float64            
 7   pv1_power_kw                 73332 non-null   float64            
 8   supplier                     483731 non-null  object             
 9   Station                      483731 non-null  int64              
 10  Date                         483

## Month by Month Summary

In [11]:
# Create the monthly financial aggregation from the minute-level dataframe
df['year_month'] = df['utc_time'].dt.to_period('M')

monthly_financials = df.groupby('year_month').agg({
    'energy_consumed_kwh': 'sum',
    'energy_produced_kwh': 'sum',
    'actual_consumption_cost_eur': 'sum',
    'actual_solar_revenue_eur': 'sum'
}).reset_index()

# Define the TLK allocation function
def get_monthly_tlk(row):
    m_str = str(row['year_month'])
    # Vattenfall fixed bracket fee spread across 5 months (€121.00 total)
    vattenfall_months = ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02']
    vattenfall_tlk = (121.00 / 5) if m_str in vattenfall_months else 0.0
    
    # Greenchoice per-kWh fee (€0.13613) applied to that month's production
    greenchoice_months = ['2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09', '2026-10']
    greenchoice_tlk = (row['energy_produced_kwh'] * 0.13613) if m_str in greenchoice_months else 0.0
    
    return vattenfall_tlk + greenchoice_tlk

monthly_financials['total_tlk_eur'] = monthly_financials.apply(get_monthly_tlk, axis=1)

# Recalculate net cost to include TLK penalties
# Note: Matching the dataframe's column name 'actual_consumption_cost_eur' and 'actual_solar_revenue_eur'
monthly_financials['final_net_cost_eur'] = (
    monthly_financials['actual_consumption_cost_eur'] + 
    monthly_financials['total_tlk_eur'] - 
    monthly_financials['actual_solar_revenue_eur']
)

# Select columns for display (mapping to correct column names)
display_columns = [
    'year_month', 
    'energy_consumed_kwh', 
    'energy_produced_kwh', 
    'actual_consumption_cost_eur', 
    'actual_solar_revenue_eur',
    'total_tlk_eur', 
    'final_net_cost_eur'
]

# Round to 2 decimal places and print
display_df = monthly_financials[display_columns].copy()
numeric_cols = display_df.select_dtypes(include=['number']).columns
display_df[numeric_cols] = display_df[numeric_cols].round(2)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
print(display_df.to_string(index=False))

year_month  energy_consumed_kwh  energy_produced_kwh  actual_consumption_cost_eur  actual_solar_revenue_eur  total_tlk_eur  final_net_cost_eur
   2025-10               223.37                 8.61                        64.10                      1.45          24.20               86.86
   2025-11               431.45                18.08                       123.72                      3.04          24.20              144.88
   2025-12               450.73                 4.53                       129.39                      0.76          24.20              152.83
   2026-01               493.70                 8.26                       141.88                      1.39          24.20              164.69
   2026-02               397.02                19.73                       108.52                      2.46          24.20              130.26
   2026-03               390.83                76.80                        87.79                      1.07          10.45               97.18

/var/folders/f3/4nxnf97x2z999nrldxm7ct1c0000gn/T/ipykernel_31578/2809185294.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['year_month'] = df['utc_time'].dt.to_period('M')


## Calculating Self-Consumption Rate



In [12]:
# Calculate self-consumed energy per row (lesser of production vs. consumption)
df['self_consumed_kwh'] = np.minimum(
    df['energy_produced_kwh'], 
    df['energy_consumed_kwh']
)

# Compute overall baseline Self-Consumption Rate (%)
total_solar_produced = df['energy_produced_kwh'].sum()
total_self_consumed = df['self_consumed_kwh'].sum()
baseline_self_consumption_rate = (total_self_consumed / total_solar_produced) * 100

print(f"Baseline Self-Consumption Rate: {baseline_self_consumption_rate:.2f}%")

Baseline Self-Consumption Rate: 25.69%


In [13]:
# Add monthly self-consumption rate column for your baseline summary tables
df['self_consumption_rate_pct'] = (
    df.groupby('year_month')['self_consumed_kwh'].transform('sum') / 
    df.groupby('year_month')['energy_produced_kwh'].transform('sum')
) * 100

In [14]:
# Save your complete baseline data with all actual costs and columns
df.to_parquet("clean_master_with_actuals.parquet")